<a href="https://colab.research.google.com/github/EshikaAbbaraju/Role_Specified_Collective_Foraging_Task_Model/blob/main/Role_Specified_Collective_Foraging_Task_Model_Code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
#!git clone https://github.com/JerryGuo2001/Role_Speficifed_Collective_Foraging_Task.git
#%cd Role_Speficifed_Collective_Foraging_Task

#tune the danger from aliens
#print out snapshot labels at the top

%cd /content/Role_Speficifed_Collective_Foraging_Task
CSV_DIR = "gridworld/"

Cloning into 'Role_Speficifed_Collective_Foraging_Task'...
remote: Enumerating objects: 305, done.
remote: Counting objects: 100% (50/50), done.
remote: Compressing objects: 100% (39/39), done.
remote: Total 305 (delta 21), reused 35 (delta 11), pack-reused 255 (from 1)
Receiving objects: 100% (305/305), 1.84 MiB | 10.71 MiB/s, done.
Resolving deltas: 100% (169/169), done.
/content/Role_Speficifed_Collective_Foraging_Task


In [9]:
from dataclasses import dataclass
from typing import Optional, Tuple, Dict, List
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter

# ----------------------------
# Environment loading
# ----------------------------
def load_env_from_csv(
    csv_path: str,
    prob_A: float = 0.8,
    prob_B: float = 0.5,
    prob_C: float = 0.2
) -> pd.DataFrame:
    raw = pd.read_csv(csv_path)
    raw = raw.rename(columns={"x": "Row", "y": "Col", "mine_type": "mine_type", "alien_id": "alien_id"})
    rows = []

    for _, row in raw.iterrows():
        r, c = int(row["Row"]), int(row["Col"])
        mine_type = str(row["mine_type"]).strip() if not pd.isna(row["mine_type"]) else ""
        alien_val = row["alien_id"]

        if mine_type == "Gold Mine A":
            reward_prob, richness = prob_A, "high"
        elif mine_type == "Gold Mine B":
            reward_prob, richness = prob_B, "medium"
        elif mine_type == "Gold Mine C":
            reward_prob, richness = prob_C, "low"
        else:
            mine_type, reward_prob, richness = "", 0.0, "none"

        alien_level = 0 if (pd.isna(alien_val) or str(alien_val).strip() == "") else int(float(alien_val))

        rows.append({
            "Row": r,
            "Col": c,
            "Location": f"{r}:{c}",
            "mine_type": mine_type,
            "richness": richness,
            "reward_prob": float(reward_prob),
            "alien_level": alien_level,
        })

    env = pd.DataFrame(rows)
    env.set_index(["Row", "Col"], inplace=True, drop=False)
    return env


# ----------------------------
# Configs
# ----------------------------
@dataclass
class ForagerConfig:
    moves_per_round: int = 5
    lambda_sec: float = 1.0
    w_scale: float = 1.0
    move_cost: float = 0.0

    mine_capacity_high: int = 8
    mine_capacity_medium: int = 5
    mine_capacity_low: int = 2

    security_pos: Optional[Tuple[int, int]] = None
    seed: Optional[int] = None

@dataclass
class SecurityConfig:
    strategy: str = "follow_forager"     # "follow_forager" | "hold_center"
    max_step_distance: int = 1
    seed: Optional[int] = None


# ----------------------------
# Forager agent
# ----------------------------
class ForagerAgent:
    def __init__(self, env: pd.DataFrame, cfg: ForagerConfig):
        self.env, self.cfg = env, cfg
        self.rng = np.random.default_rng(cfg.seed)

        max_r = int(self.env.index.get_level_values(0).max())
        max_c = int(self.env.index.get_level_values(1).max())
        self.pos = (max_r // 2, max_c // 2)
        self.security_pos = cfg.security_pos or (max_r // 2, max_c // 2)

        self.t, self.total_reward = 0, 0.0
        self.visited = {self.pos}
        self.stunned_steps = 0
        self.print_snapshot_actions = True

        self.log, self.frames_pos, self.frames_action, self.frames_decision = [], [], [], []
        self.frames_Vdig, self.frames_Vmove, self.frames_w, self.frames_total_reward = [], [], [], []
        self.frames_depleted = []

        self.mine_remaining = {}
        for (r, c), row in self.env.iterrows():
            mine_type = str(row["mine_type"]).strip()
            if mine_type == "Gold Mine A":
                self.mine_remaining[(r, c)] = int(self.cfg.mine_capacity_high)
            elif mine_type == "Gold Mine B":
                self.mine_remaining[(r, c)] = int(self.cfg.mine_capacity_medium)
            elif mine_type == "Gold Mine C":
                self.mine_remaining[(r, c)] = int(self.cfg.mine_capacity_low)

    def current_reward_prob(self) -> float:
        if self.pos in self.mine_remaining and self.mine_remaining[self.pos] <= 0:
            return 0.0
        return float(self.env.loc[self.pos, "reward_prob"])

    def neighbors(self) -> List[Tuple[int, int]]:
        r, c = self.pos
        return [p for p in [(r - 1, c), (r + 1, c), (r, c - 1), (r, c + 1)] if p in self.env.index]

    def Vdig(self) -> float:
        return self.current_reward_prob()

    def Vmove(self) -> float:
        return self.total_reward / float(self.t) if self.t > 0 else 0.0

    def w_t(self, Vdig_t: float, Vmove_t: float) -> float:
        delta = Vdig_t - Vmove_t
        return float(1.0 / (1.0 + np.exp(-max(self.cfg.w_scale, 1e-6) * delta)))

    def E_exploit(self, s_prime: Tuple[int, int]) -> float:
        known = [p for p in self.visited if (p in self.env.index) and (self.env.loc[p, "mine_type"] != "")]
        if not known:
            return 0.0
        d_min = min(abs(s_prime[0] - p[0]) + abs(s_prime[1] - p[1]) for p in known)
        return max(float(self.env.loc[p, "reward_prob"]) for p in known) / (1.0 + d_min)

    def E_explore(self, s_prime: Tuple[int, int]) -> float:
        unv = [p for p in self.env.index if p not in self.visited]
        if not unv:
            return 0.0
        d_min = min(abs(s_prime[0] - p[0]) + abs(s_prime[1] - p[1]) for p in unv)
        return 1.0 / (1.0 + d_min)

    def A_goal(self, s_prime: Tuple[int, int], w: float) -> float:
        return (1.0 - w) * self.E_exploit(s_prime) + w * self.E_explore(s_prime)

    def _log(self, **kw):
        kw.setdefault("step", self.t)
        kw.setdefault("row", self.pos[0])
        kw.setdefault("col", self.pos[1])
        kw.setdefault("total_reward", self.total_reward)
        self.log.append(kw)

    def _snapshot(self, action: str, decision: str, Vdig: float, Vmove: float, w: float):
        self.frames_pos.append(tuple(self.pos))
        self.frames_action.append(action)
        self.frames_decision.append(decision)
        self.frames_Vdig.append(float(Vdig))
        self.frames_Vmove.append(float(Vmove))
        self.frames_w.append(float(w))
        self.frames_total_reward.append(float(self.total_reward))
        self.frames_depleted.append({p for p, rem in self.mine_remaining.items() if rem <= 0})

        if self.print_snapshot_actions:
            print(
                f"[snapshot] step={self.t} action={action} decision={decision} "
                f"pos=({self.pos[0]},{self.pos[1]}) total_reward={self.total_reward:.3f}"
            )

    def _dig(self) -> float:
        r = self.current_reward_prob()
        self.total_reward += r
        if self.pos in self.mine_remaining and self.mine_remaining[self.pos] > 0:
            self.mine_remaining[self.pos] -= 1
        return r

    def _move_to_best_A(self, w: float) -> Tuple[bool, int]:
        nbrs = self.neighbors()
        if not nbrs:
            return False, 0

        best_val, best_p = -np.inf, None
        for p in nbrs:
            a = self.A_goal(p, w)
            d = abs(p[0] - self.security_pos[0]) + abs(p[1] - self.security_pos[1])
            v = self.cfg.lambda_sec * float(d) * a - self.cfg.move_cost
            if v > best_val:
                best_val, best_p = v, p

        if best_p is None:
            return False, 0

        self.pos = best_p
        self.visited.add(self.pos)

        al = int(self.env.loc[self.pos, "alien_level"])
        # If on alien and not protected by security, stunned for 3 moves
        if al > 0 and self.pos != self.security_pos:
            self.stunned_steps = 3
        return True, al

    def step(self):
        if self.stunned_steps > 0:
            self._log(action="stunned", decision="stunned", Vdig=np.nan, Vmove=np.nan, w=np.nan)
            self._snapshot("stunned", "stunned", np.nan, np.nan, np.nan)
            self.stunned_steps -= 1
            self.t += 1
            return

        if self.current_reward_prob() > 0.0:
            Vd, Vm = self.Vdig(), self.Vmove()
            w = self.w_t(Vd, Vm)
            r_gained = self._dig()
            self._log(action="dig", decision="dig", Vdig=Vd, Vmove=Vm, w=w, r_gained=r_gained)
            self._snapshot("dig", "dig", Vd, Vm, w)
            self.t += 1
            return

        Vd, Vm = self.Vdig(), self.Vmove()
        w = self.w_t(Vd, Vm)

        force_move = (
            self.current_reward_prob() == 0.0
            and (any(self.env.loc[p, "reward_prob"] > 0 for p in self.neighbors())
                 or any(p not in self.visited for p in self.env.index))
        )

        if (Vd > Vm) and not force_move:
            self._dig()
            self._log(action="dig", decision="dig", Vdig=Vd, Vmove=Vm, w=w)
            self._snapshot("dig", "dig", Vd, Vm, w)
        else:
            moved, alien_lvl = self._move_to_best_A(w)
            self._log(
                action="move" if moved else "no_move",
                decision="move",
                Vdig=Vd, Vmove=Vm, w=w,
                alien_level=alien_lvl if alien_lvl else None
            )
            self._snapshot("move" if moved else "no_move", "move", Vd, Vm, w)

        self.t += 1


# ----------------------------
# Security agent (integrated scaffold)
# ----------------------------
class SecurityAgent:
    def __init__(self, env: pd.DataFrame, cfg: SecurityConfig):
        self.env, self.cfg = env, cfg
        max_r = int(self.env.index.get_level_values(0).max())
        max_c = int(self.env.index.get_level_values(1).max())
        self.pos = (max_r // 2, max_c // 2)
        self.log = []
        self.frames_pos = [self.pos]

    def _neighbors_of(self, pos: Tuple[int, int]) -> List[Tuple[int, int]]:
        r, c = pos
        return [p for p in [(r - 1, c), (r + 1, c), (r, c - 1), (r, c + 1)] if p in self.env.index]

    def _step_toward(self, target: Tuple[int, int]):
        if self.pos == target:
            return
        nbrs = self._neighbors_of(self.pos)
        if not nbrs:
            return
        best = min(nbrs, key=lambda p: abs(p[0] - target[0]) + abs(p[1] - target[1]))
        self.pos = best

    def step(self, t: int, forager_pos: Tuple[int, int]):
        if self.cfg.strategy == "hold_center":
            pass
        else:  # follow_forager
            self._step_toward(forager_pos)

        self.log.append({"step": t, "security_row": self.pos[0], "security_col": self.pos[1]})
        self.frames_pos.append(self.pos)


# ----------------------------
# Multi-round simulation wrapper
# ----------------------------
def run_foraging_simulation(
    csv_path: str,
    rounds: int = 3,
    forager_cfg: Optional[ForagerConfig] = None,
    security_cfg: Optional[SecurityConfig] = None,
    seed: int = 0,
    out_prefix: str = "forager",
    save_csv: bool = True,
    save_gif: bool = True
) -> Dict[str, object]:
    """
    Runs multiple rounds; each round has forager_cfg.moves_per_round moves.
    Security agent is stepped each move and updates forager.security_pos.
    """
    env = load_env_from_csv(csv_path)

    forager_cfg = forager_cfg or ForagerConfig(seed=seed)
    security_cfg = security_cfg or SecurityConfig(seed=seed)

    # Force total moves from rounds * moves_per_round
    total_moves = rounds * forager_cfg.moves_per_round
    forager_cfg = ForagerConfig(
        moves_per_round=forager_cfg.moves_per_round,
        lambda_sec=forager_cfg.lambda_sec,
        w_scale=forager_cfg.w_scale,
        move_cost=forager_cfg.move_cost,
        mine_capacity_high=forager_cfg.mine_capacity_high,
        mine_capacity_medium=forager_cfg.mine_capacity_medium,
        mine_capacity_low=forager_cfg.mine_capacity_low,
        security_pos=forager_cfg.security_pos,
        seed=forager_cfg.seed,
    )

    forager = ForagerAgent(env, forager_cfg)
    security = SecurityAgent(env, security_cfg)
    forager.security_pos = security.pos

    for t in range(total_moves):
        # Security updates first, then forager reacts with latest security position
        security.step(t=t, forager_pos=forager.pos)
        forager.security_pos = security.pos
        forager.step()

    log_df = pd.DataFrame(forager.log)
    sec_log_df = pd.DataFrame(security.log)

    if save_csv:
        log_df.to_csv(f"{out_prefix}_log.csv", index=False)
        sec_log_df.to_csv(f"{out_prefix}_security_log.csv", index=False)

    if save_gif:
        animate_forager(env, forager, security_agent=security, outpath=f"{out_prefix}.gif")

    return {
        "env": env,
        "forager": forager,
        "security": security,
        "forager_log": log_df,
        "security_log": sec_log_df,
        "total_reward": forager.total_reward,
        "rounds": rounds,
        "moves_per_round": forager_cfg.moves_per_round,
        "total_moves": total_moves,
    }


# ----------------------------
# Animation (forager + optional security)
# ----------------------------
def animate_forager(
    env: pd.DataFrame,
    agent: ForagerAgent,
    security_agent: Optional[SecurityAgent] = None,
    outpath: str = "forager.gif"
):
    env_idx = env.set_index(["Row", "Col"], drop=False)
    max_r, max_c = int(env_idx["Row"].max()), int(env_idx["Col"].max())

    base = np.zeros((max_r + 1, max_c + 1))
    for (r, c), row in env_idx.iterrows():
        base[r, c] = row["reward_prob"]

    fig, ax = plt.subplots(figsize=(6, 6))
    cmap = plt.cm.YlOrRd.copy()
    cmap.set_bad(color="lightgray")
    im = ax.imshow(base, cmap=cmap, origin="upper", vmin=0, vmax=1)

    forager_dot, = ax.plot([], [], "bo", markersize=9, label="Forager")
    sec_dot = None
    if security_agent is not None:
        sec_dot, = ax.plot([], [], "gs", markersize=8, label="Security")

    ax.set_xticks(range(max_c + 1))
    ax.set_yticks(range(max_r + 1))
    ax.grid(True, linestyle=":", alpha=0.3)
    ax.legend(loc="upper right")

    def init():
        forager_dot.set_data([], [])
        if sec_dot is not None:
            sec_dot.set_data([], [])
        ax.set_title("Forager/Security")
        artists = [im, forager_dot]
        if sec_dot is not None:
            artists.append(sec_dot)
        return artists

    def update(f):
        frame_grid = base.copy()
        if f < len(agent.frames_depleted):
            for dr, dc in agent.frames_depleted[f]:
                frame_grid[dr, dc] = np.nan
        im.set_data(frame_grid)

        fr, fc = agent.frames_pos[f]
        forager_dot.set_data([fc], [fr])

        title = (
            f"step={f} action={agent.frames_action[f]} "
            f"decision={agent.frames_decision[f]} "
            f"reward={agent.frames_total_reward[f]:.3f}"
        )

        artists = [im, forager_dot]

        if security_agent is not None and f < len(security_agent.frames_pos):
            sr, sc = security_agent.frames_pos[f]
            sec_dot.set_data([sc], [sr])
            title += f" | security=({sr},{sc})"
            artists.append(sec_dot)

        ax.set_title(title)
        return artists

    anim = FuncAnimation(fig, update, init_func=init, frames=len(agent.frames_pos), interval=400, blit=True)
    anim.save(outpath, writer=PillowWriter(fps=4))
    plt.close(fig)

In [11]:
single_csv = os.path.join(CSV_DIR, "middle_reward_middle_risk_01.csv")
result = run_foraging_simulation(
     csv_path=single_csv,
     rounds=4,
     forager_cfg=ForagerConfig(moves_per_round=5, mine_capacity_high=8, mine_capacity_medium=5, mine_capacity_low=2, seed=0),
     security_cfg=SecurityConfig(strategy="follow_forager"),
     out_prefix="forager_r4",
     save_csv=True,
     save_gif=True
 )
print("Total reward:", result["total_reward"])
print(result["forager_log"][["step", "action", "decision", "row", "col", "total_reward"]].to_string(index=False))

[snapshot] step=0 action=move decision=move pos=(5,6) total_reward=0.000
[snapshot] step=1 action=move decision=move pos=(4,6) total_reward=0.000
[snapshot] step=2 action=move decision=move pos=(3,6) total_reward=0.000
[snapshot] step=3 action=move decision=move pos=(2,6) total_reward=0.000
[snapshot] step=4 action=move decision=move pos=(1,6) total_reward=0.000
[snapshot] step=5 action=move decision=move pos=(0,6) total_reward=0.000
[snapshot] step=6 action=move decision=move pos=(0,5) total_reward=0.000
[snapshot] step=7 action=move decision=move pos=(1,5) total_reward=0.000
[snapshot] step=8 action=move decision=move pos=(2,5) total_reward=0.000
[snapshot] step=9 action=dig decision=dig pos=(2,5) total_reward=0.200
[snapshot] step=10 action=dig decision=dig pos=(2,5) total_reward=0.400
[snapshot] step=11 action=move decision=move pos=(3,5) total_reward=0.400
[snapshot] step=12 action=dig decision=dig pos=(3,5) total_reward=0.600
[snapshot] step=13 action=dig decision=dig pos=(3,5) t